# Week 5 Mini Project — Compact STT Demo

This notebook runs a pretrained ASR pipeline on a tiny audio sample to show end-to-end inference (audio → text). It also contains a short commented skeleton showing how you would set up fine-tuning with `Trainer` and `TrainingArguments`. The inference uses `facebook/wav2vec2-base-960h` which is CPU-runnable on Colab but will download model weights (~100s of MB). Run the notebook cells one-by-one.

In [1]:
# Install runtime dependencies
!pip install -q transformers datasets soundfile librosa

In [3]:
from transformers import pipeline
from datasets import load_dataset, Audio, Dataset
import numpy as np
import soundfile as sf
import librosa

# Load a tiny audio sample (same robust fallback as other notebooks)
try:
    ds = load_dataset('mozilla-foundation/common_voice_13_0', 'en', split='train[:0.1%]')
    ds_name = 'common_voice_13_0'
except Exception as e:
    print('Common Voice load failed, using librispeech demo validation split:', e)
    try:
        ds = load_dataset('hf-internal-testing/librispeech_asr_demo', split='validation')
        ds_name = 'librispeech_asr_demo'
    except Exception as e2:
        print('Fallback librispeech demo failed:', e2)
        # Final fallback: synthetic 1s tone
        sr = 16000
        duration = 1.0
        t = np.linspace(0, duration, int(sr * duration), endpoint=False)
        arr = (0.05 * np.sin(2 * np.pi * 440 * t)).astype('float32')
        example = {'audio': {'array': arr, 'sampling_rate': sr}, 'text': 'synthetic tone'}
        ds = Dataset.from_list([example])
        ds_name = 'synthetic'

if ds_name != 'synthetic':
    ds = ds.cast_column('audio', Audio(decode=False))
    example = ds[0]
else:
    example = ds[0]

# Extract audio safely
arr = None
sr = 16000
if 'audio' in example:
    audio = example['audio']
    if isinstance(audio, dict) and 'array' in audio:
        arr = np.array(audio['array']).astype('float32')
        sr = int(audio.get('sampling_rate', 16000))
    elif isinstance(audio, dict) and 'path' in audio:
        path = audio['path']
        try:
            arr, sr = sf.read(path)
            arr = np.array(arr).astype('float32')
        except Exception as e_path:
            print('soundfile path read failed; trying librosa:', e_path)
            try:
                arr, sr = librosa.load(path, sr=None)
                arr = np.array(arr).astype('float32')
            except Exception as e_lib:
                print('librosa path read failed; falling back to synthetic:', e_lib)
    else:
        try:
            arr, sr = sf.read(audio)
            arr = np.array(arr).astype('float32')
        except Exception:
            sr = 16000
            duration = 1.0
            t = np.linspace(0, duration, int(sr * duration), endpoint=False)
            arr = (0.05 * np.sin(2 * np.pi * 440 * t)).astype('float32')
else:
    sr = 16000
    duration = 1.0
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    arr = (0.05 * np.sin(2 * np.pi * 440 * t)).astype('float32')

if arr is None:
    sr = 16000
    duration = 1.0
    t = np.linspace(0, duration, int(sr * duration), endpoint=False)
    arr = (0.05 * np.sin(2 * np.pi * 440 * t)).astype('float32')

# Ensure 16k sampling rate for Wav2Vec2 pipeline
if sr != 16000:
    arr = librosa.resample(arr, orig_sr=sr, target_sr=16000)
    sr = 16000

print('Sample rate used for inference:', sr, 'duration s:', len(arr)/sr, 'source:', ds_name)

# Run ASR pipeline (this will download the pretrained model weights)
asr = pipeline('automatic-speech-recognition', model='facebook/wav2vec2-base-960h')
print('Running inference on one sample...')
result = asr(arr, chunk_length_s=10)
print('Transcription:')
print(result['text'])

# Small note: for fine-tuning you would prepare dataset -> processor inputs -> Trainer.
# Below is a commented skeleton (do not run in this demo).




Repo card metadata block was not found. Setting CardData to empty.


Common Voice load failed, using librispeech demo validation split: The directory at hf://datasets/mozilla-foundation/common_voice_13_0@ff2bbb54dcdb597100fe534a1b911ff9103f9e22 doesn't contain any data files
soundfile path read failed; trying librosa: Error opening '1272-128104-0000.flac': System error.


/tmp/ipython-input-3909994199.py:49: UserWarning: PySoundFile failed. Trying audioread instead.
  arr, sr = librosa.load(path, sr=None)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


librosa path read failed; falling back to synthetic: [Errno 2] No such file or directory: '1272-128104-0000.flac'
Sample rate used for inference: 16000 duration s: 1.0 source: librispeech_asr_demo


model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-base-960h and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


Running inference on one sample...
Transcription:

